import libraries

In [1]:
import os
import json
import certifi
import numpy
import pandas as pd
import pymongo
from sqlalchemy import create_engine, text

In [2]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "#Hi10172004"
 
src_dbname = "sakila"
dst_dbname = "sakila_dw"
 
mysql_args = {
    "uid"      : user_id,
    "pwd"      : pwd,
    "hostname" : host_name,
    "dbname"   : dst_dbname
}
 
mongodb_args = {
    "user_name"        : "",
    "password"         : "",
    "cluster_name"     : "",
    "cluster_subnet"   : "",
    "cluster_location" : "local", # or could be Atlas
    "db_name"          : "sakila_catalog"
}

In [3]:
data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)

In [4]:
#get data from mysql
def get_dataframe(user_id, pwd, host_name, db_name, sql_query):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(sql_query, connection)
    connection.close()
    return dframe

#set data in new db
def set_dataframe(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
 
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
            db_connection.commit()
 
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')
            db_connection.commit()
 
    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
 
    db_connection.close()
    
def get_mongo_client(**args):
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
 
    else:
        if args["cluster_location"] == "atlas":
            connect_str  = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
 
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
 
    return client

def get_mongo_dataframe(mongo_client, db_name, collection, query):
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    return dframe

def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
 
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
 
    mongo_client.close()

In [5]:
conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()
 
connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE {dst_dbname};"))
connection.commit()
connection.close()
 
print(f"Database '{dst_dbname}' created.")

Database 'sakila_dw' created.


In [6]:
#moving payments into .csv
sql_payments = "SELECT * FROM sakila.payment;"
df_payments_export = get_dataframe(user_id, pwd, host_name, src_dbname, sql_payments)
 
csv_file = os.path.join(data_dir, 'sakila_payments.csv')
df_payments_export.to_csv(csv_file, index=False)
print("Export successful")

Exported 16,044 payment rows to CSV.


In [8]:
sql_films_export = """
    SELECT
        f.film_id,
        f.title,
        f.description,
        f.release_year,
        f.rental_duration,
        f.rental_rate,
        f.length            AS film_length_minutes,
        f.replacement_cost,
        f.rating,
        f.special_features,
        cat.name            AS category
    FROM sakila.film          AS f
    JOIN sakila.film_category AS fc  ON f.film_id      = fc.film_id
    JOIN sakila.category      AS cat ON fc.category_id = cat.category_id;
"""
df_films_export = get_dataframe(user_id, pwd, host_name, src_dbname, sql_films_export)
 
# Save as JSON
json_file = os.path.join(data_dir, 'sakila_films.json')
df_films_export.to_json(json_file, orient='records', indent=2)
print(f"Exported {len(df_films_export):,} film documents to JSON.")
 
# Load into MongoDB using set function
client = get_mongo_client(**mongodb_args)
 
json_files = {"films" : 'sakila_films.json'}
 
set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)
print("Film documents loaded into MongoDB.")

Exported 1,000 film documents to JSON.


ServerSelectionTimeoutError: localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69c1fe4da34269858df3c910, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>